# 7. Sync to Lakebase (app serving)

Mirrors the Delta tables the app reads into Lakebase as **synced** tables
(SNAPSHOT, one-way Delta→Postgres) and creates the **native** writable
`reviews` table the app appends corrections to. Runs last in the pipeline so
the app always serves the freshest predictions.

Read-only synced mirrors: invoices, predictions, schema_registry, the three
taxonomies, ground_truth. Native (app writes): reviews.

Idempotent: existing synced tables are left in place (SNAPSHOT refreshes
them); the native table is created if missing.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config

spark = get_spark()
config = load_config()

dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
spark.sql(f'USE {catalog}.{schema}')

## Native writable `reviews` table + connection

The app appends human review events here (it is **not** a synced mirror, so
the app can write to it). Schema matches the Delta `reviews` table.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()
SP_ROLE = 'aab57292-8e82-48bf-8e43-e8f43fe2743f'  # spend-app service principal

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(host=inst.read_write_dns, dbname=config.lakebase_dbname,
                            user=user, password=cred.token, sslmode='require')

with pg_connect() as conn, conn.cursor() as cur:
    cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}"')
    cur.execute(f'''
        CREATE TABLE IF NOT EXISTS "{schema}".reviews (
            review_id TEXT PRIMARY KEY,
            order_id TEXT, schema_name TEXT, reviewed_source TEXT,
            action TEXT, code TEXT, label TEXT, level_path TEXT[],
            reviewer TEXT, comment TEXT, created_at TIMESTAMPTZ
        )''')
    try:
        cur.execute(f'GRANT USAGE ON SCHEMA "{schema}" TO "{SP_ROLE}"')
        cur.execute(f'GRANT SELECT, INSERT ON "{schema}".reviews TO "{SP_ROLE}"')
    except Exception as e:
        print(f'grant skipped: {e}')
    conn.commit()
print('native reviews table ready')

## Create synced (read-only) mirrors

One Databricks synced table per Delta table the app reads. SNAPSHOT mode
refreshes on each run. Existing ones are skipped.

In [ ]:
from databricks.sdk.service.database import (
    SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec, SyncedTableSchedulingPolicy,
)

instance_name = lakebase_instance
TABLES_TO_SYNC = [
    {'source': 'invoices',            'pk': ['order_id']},
    {'source': 'predictions',         'pk': ['order_id', 'schema_name', 'source']},
    {'source': 'ground_truth',        'pk': ['order_id', 'schema_name']},
    {'source': 'schema_registry',     'pk': ['name']},
    {'source': 'three_level_taxonomy','pk': ['code']},
    {'source': 'gl_map_taxonomy',     'pk': ['code']},
    {'source': 'unspsc_taxonomy',     'pk': ['code']},
]

for t in TABLES_TO_SYNC:
    src_full = f'{catalog}.{schema}.{t["source"]}'
    sync_full = f'{catalog}.{schema}.{t["source"]}_synced'
    print(f'Syncing {src_full} -> {sync_full}')
    try:
        existing = w.database.get_synced_database_table(name=sync_full)
        print(f'  exists ({existing.data_synchronization_status.detailed_state})')
        continue
    except Exception:
        pass
    try:
        w.database.create_synced_database_table(SyncedDatabaseTable(
            name=sync_full, database_instance_name=instance_name,
            logical_database_name=config.lakebase_dbname,
            spec=SyncedTableSpec(
                source_table_full_name=src_full, primary_key_columns=t['pk'],
                scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
                create_database_objects_if_missing=True,
                new_pipeline_spec=NewPipelineSpec(storage_catalog=catalog, storage_schema=schema),
            )))
        print('  created')
    except Exception as e:
        print(f'  error: {e}')